# Defense C distillation: small student mimicking the layered defense

**Capstone: Prompt-Injection Defense Evaluation — Notebook 09**

Trains a small classifier (DistilBERT, 66M params) to mimic Defense C's binary output (Defense A OR Defense B). Compares the distilled student against:

1. **Defense C labels** (teacher) — upper bound on what the student can learn
2. **True eval-set labels** — how well the student approximates the layered defense's actual behavior
3. **Off-the-shelf DeBERTa baseline** — does the distilled student beat the same-architecture-class baseline?

## Why distillation

Defense C requires two model calls per prompt: DeBERTa (Defense A, local, fast) AND Sonnet 4.6 (Defense B, API, ~2-3 sec, $0.004 per call). A successful distillation collapses both into a single ~50ms DistilBERT inference at near-zero per-call cost. The trade-off is whether the student preserves Defense C's catch rate.

This is the DS4 §6.10 stretch goal and a direct §7 deployment artifact.

## Required artifacts (upload to Drive)

- `results/eval_set.parquet` (4,546 rows)
- `results/defense_a_full_eval_set.csv` (Defense A predictions on all 4,546 rows)
- `results/defense_b_full.csv` (Defense B v1.21 verdicts on all 4,546 rows)
- `results/eval_set_splits.parquet` (from notebook 08; reuse the SAME train/val/test split for apples-to-apples)

## Hardware: T4 or L4 GPU + High-RAM

## 1. Environment setup

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/capstone_lora')
DATA_DIR = DRIVE_ROOT / 'data'
EVAL_SET_PATH = DATA_DIR / 'eval_set.parquet'
SPLITS_PATH = DATA_DIR / 'eval_set_splits.parquet'
DEFENSE_A_PATH = DATA_DIR / 'defense_a_full_eval_set.csv'
DEFENSE_B_PATH = DATA_DIR / 'defense_b_full.csv'

ADAPTER_DIR = DRIVE_ROOT / 'adapters' / 'distilbert_defense_c_student_v1'
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = DRIVE_ROOT / 'results' / 'distillation_metrics.json'
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

for p in [EVAL_SET_PATH, SPLITS_PATH, DEFENSE_A_PATH, DEFENSE_B_PATH]:
    print(f'  {p.name}: {"OK" if p.exists() else "MISSING — upload before continuing"}')

In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'

# Remove pre-installed torchao that conflicts with latest peft (we don't use it)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '--quiet', 'torchao'], check=False)
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'transformers>=4.53', 'datasets', 'accelerate', 'scikit-learn',
    'tqdm', 'matplotlib', 'sentencepiece',
])
print('Packages installed.')

try:
    from google.colab import userdata
    from huggingface_hub import login
    tok = userdata.get('HF_TOKEN')
    if tok:
        login(token=tok)
        print('Logged in to HuggingFace.')
except Exception as e:
    print(f'HF login skipped: {e}')

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import torch
from scipy.stats import binomtest
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, cohen_kappa_score
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    USE_BF16 = cc[0] >= 8
    USE_FP16 = not USE_BF16
    print(f'GPU: {torch.cuda.get_device_name(0)}; precision: {"bf16" if USE_BF16 else "fp16"}')
else:
    USE_BF16, USE_FP16 = False, False
    print('No GPU detected; training will be slow.')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 2. Build Defense C labels from Defense A + Defense B predictions

Defense C = OR-gate: a prompt is flagged if either Defense A flags it OR Defense B's judge returns HIJACKED (or AMBIGUOUS, collapsed conservatively per §3.2 v1.21 convention).

Joins on `prompt_idx`. The resulting `defense_c_label` is the **teacher signal** the distilled student tries to match.

In [ ]:
eval_set = pd.read_parquet(EVAL_SET_PATH)
splits = pd.read_parquet(SPLITS_PATH)
defense_a = pd.read_csv(DEFENSE_A_PATH)
defense_b = pd.read_csv(DEFENSE_B_PATH)

print(f'eval_set: {len(eval_set):,} rows')
print(f'splits:   {len(splits):,} rows')
print(f'defense_a columns: {[c for c in defense_a.columns if "pred" in c.lower() or "label" in c.lower()][:5]}')
print(f'defense_b columns: {[c for c in defense_b.columns if "verdict" in c.lower() or "hijack" in c.lower()][:5]}')

In [ ]:
# Identify Defense A prediction column
a_candidates = [c for c in defense_a.columns if 'pred' in c.lower() and 'deberta' in c.lower()]
if not a_candidates:
    a_candidates = [c for c in defense_a.columns if c.lower() in ('deberta_pred', 'pred', 'protectai_pred')]
A_PRED_COL = a_candidates[0]

# Identify Defense B verdict column (sonnet_verdict_v121 is the canonical one)
B_VERDICT_COL = 'sonnet_verdict_v121' if 'sonnet_verdict_v121' in defense_b.columns else None
assert B_VERDICT_COL is not None, f'No sonnet_verdict_v121 in defense_b; columns: {list(defense_b.columns)}'

print(f'Defense A column: {A_PRED_COL}')
print(f'Defense B column: {B_VERDICT_COL}')

# Build a single DataFrame with prompt, true label, Defense A pred, Defense B verdict, Defense C derived
df = eval_set[['prompt_idx', 'dataset', 'prompt', 'label']].copy()
df = df.merge(defense_a[['prompt_idx', A_PRED_COL]], on='prompt_idx', how='left')
df = df.merge(defense_b[['prompt_idx', B_VERDICT_COL]], on='prompt_idx', how='left')

df = df.rename(columns={A_PRED_COL: 'def_a_pred', B_VERDICT_COL: 'def_b_verdict'})
df['def_b_pred'] = (df['def_b_verdict'].astype(str).str.upper() != 'CLEAN').astype(int)  # conservative collapse
df['defense_c_label'] = ((df['def_a_pred'] == 1) | (df['def_b_pred'] == 1)).astype(int)

print(f'\nDefense C label distribution: {dict(df["defense_c_label"].value_counts())}')
print(f'Of true positives (n={int((df["label"]==1).sum())}), Defense C flags: {int(((df["label"]==1) & (df["defense_c_label"]==1)).sum())}')
print(f'Of true negatives (n={int((df["label"]==0).sum())}), Defense C flags: {int(((df["label"]==0) & (df["defense_c_label"]==1)).sum())}')

In [ ]:
# Join the split assignment from notebook 08
df = df.merge(splits[['prompt_idx', 'split']], on='prompt_idx', how='left')
assert df['split'].notna().all(), 'Some rows have no split assignment'
print('Split sizes (joined):')
print(df['split'].value_counts())
print('\nDefense C label rate per split:')
print(df.groupby('split')['defense_c_label'].agg(['count', 'mean']))

## 3. Distill: train DistilBERT to predict Defense C output

**Student:** `distilbert-base-uncased` (66M params, ~6x smaller than DeBERTa-v3-base, ~4x faster inference).

**Training labels:** `defense_c_label` (not the true eval-set label). The student learns to mimic the teacher's binary output. This is hard-label distillation; soft-label distillation (KL divergence on logits) would require Defense C to expose probabilities, which the LLM-as-judge does not.

**Recipe:** full fine-tune (not LoRA — DistilBERT is small enough that full FT is cheap), 3 epochs, lr=5e-5.

### Modeling methodology note: fixed clean recipe, no hyperparameter tuning

This experiment uses ONE configuration with sensible defaults: full fine-tune (not LoRA — DistilBERT is small enough), 3 epochs, lr=5e-5, batch_size=32, max_length=512. We deliberately did NOT sweep hyperparameters. The goal is to establish a clean baseline that demonstrates whether distillation works, not to find the optimal recipe.

**In a deployment context, all of these would be candidates for tuning:**

| Hyperparameter | Typical sweep range | Notes |
|---|---|---|
| Student model | DistilBERT / TinyBERT / MiniLM / DeBERTa-base | Smaller = faster, less capacity |
| Distillation objective | hard labels (here) / soft labels with KL divergence / hard+soft mixture | Soft labels need teacher to expose probabilities, which Defense C's judge does not |
| Learning rate | 1e-5 to 1e-4 | Lower for smaller students |
| Num epochs | 2 to 10 | Risk of overfit at higher |
| Batch size | 16 to 64 | Memory vs noise trade-off |
| Class weighting | balanced / unbalanced | Defense C label is highly imbalanced (~93% flagged); class weighting could help calibration |
| Temperature (for soft-label distill) | 1.0 to 4.0 | Smooths teacher logits |
| Adapter (LoRA on student) | optional | Not needed at DistilBERT scale; useful for larger students |

The Cohen kappa and student-vs-true-label F1 reported below are conservative baselines. Production deployment of a distilled-defender would benefit from a focused sweep; here we report the result of the recipe as specified.

In [ ]:
STUDENT_MODEL = 'distilbert-base-uncased'
print(f'Loading {STUDENT_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL,
    num_labels=2,
    id2label={0: 'CLEAN', 1: 'FLAGGED'},
    label2id={'CLEAN': 0, 'FLAGGED': 1},
)
n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')

In [ ]:
def to_hf_dataset(d: pd.DataFrame, label_col: str) -> Dataset:
    return Dataset.from_pandas(
        d[['prompt', label_col]].rename(columns={label_col: 'labels'}).reset_index(drop=True)
    )

train_ds = to_hf_dataset(df[df['split']=='train'], 'defense_c_label')
val_ds = to_hf_dataset(df[df['split']=='val'], 'defense_c_label')
test_ds = to_hf_dataset(df[df['split']=='test'], 'defense_c_label')

def tokenize(batch):
    return tokenizer(batch['prompt'], truncation=True, max_length=512, padding=False)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=['prompt'])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=['prompt'])
test_tok = test_ds.map(tokenize, batched=True, remove_columns=['prompt'])
print(f'Tokenized: train={len(train_tok)}, val={len(val_tok)}, test={len(test_tok)}')

In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', pos_label=1, zero_division=0)
    return {'accuracy': accuracy_score(labels, preds), 'precision': p, 'recall': r, 'f1': f1}

training_args = TrainingArguments(
    output_dir='/content/distill_output',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    lr_scheduler_type='linear',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    fp16=USE_FP16, bf16=USE_BF16,
    seed=SEED, report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_f1',
    greater_is_better=True,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_tok, eval_dataset=val_tok,
    data_collator=collator, compute_metrics=compute_metrics,
)

In [ ]:
print('Distilling DistilBERT to mimic Defense C (3 epochs)...')
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\nTraining done in {elapsed/60:.1f} min.')

## 4. Evaluate the student: vs Defense C labels AND vs TRUE labels

The student was trained to match Defense C's output. Two important comparisons on the held-out test split:

1. **Student vs Defense C (teacher)** — how faithfully did distillation reproduce the teacher? Cohen's kappa is the right metric here (we're measuring rater agreement, not classification accuracy).
2. **Student vs TRUE label** — how does the student perform on the actual prompt-injection task? Compared to Defense C's own performance on TRUE labels, does the student preserve, lose, or gain?

In [ ]:
def wilson_ci(s, n, alpha=0.05):
    if n == 0:
        return (0.0, 1.0)
    r = binomtest(int(s), int(n))
    lo, hi = r.proportion_ci(confidence_level=1 - alpha, method='wilson')
    return float(lo), float(hi)

def per_class_metrics(y_true, y_pred, label_name=''):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    rec_ci = wilson_ci(tp, tp + fn) if (tp + fn) > 0 else (0.0, 0.0)
    prec_ci = wilson_ci(tp, tp + fp) if (tp + fp) > 0 else (0.0, 0.0)
    return {'label': label_name, 'n': len(y_true), 'n_pos': int((y_true == 1).sum()),
            'precision': float(p), 'precision_ci': prec_ci,
            'recall': float(r), 'recall_ci': rec_ci,
            'f1': float(f), 'accuracy': float(accuracy_score(y_true, y_pred))}

def print_metrics_table(ml, title):
    print(f'\n=== {title} ===')
    print(f'{"Slice":<14} {"n":>5} {"n+":>5} {"Prec":>6} {"Recall":>7} {"F1":>6} {"Acc":>6}')
    for m in ml:
        print(f'{m["label"]:<14} {m["n"]:>5} {m["n_pos"]:>5} {m["precision"]:>6.3f} {m["recall"]:>7.3f} {m["f1"]:>6.3f} {m["accuracy"]:>6.3f}')

In [ ]:
# Predict on test
test_df = df[df['split']=='test'].reset_index(drop=True)
preds = trainer.predict(test_tok)
student_pred = np.argmax(preds.predictions, axis=-1)
test_df['student_pred'] = student_pred

# (1) Student vs teacher (Defense C) — agreement
kappa_student_teacher = cohen_kappa_score(test_df['defense_c_label'], test_df['student_pred'])
agree_student_teacher = (test_df['defense_c_label'] == test_df['student_pred']).mean()
print(f'STUDENT vs DEFENSE C (teacher) on test split (n={len(test_df)}):')
print(f'  Agreement: {agree_student_teacher:.3f}')
print(f'  Cohen kappa: {kappa_student_teacher:.3f}')

In [ ]:
# (2) Student vs TRUE label, with per-dataset breakdown
student_metrics = [per_class_metrics(test_df['label'], test_df['student_pred'], 'overall')]
for ds in ['deepset', 'neuralchemy', 'spml']:
    sub = test_df[test_df['dataset']==ds]
    if len(sub) > 0:
        student_metrics.append(per_class_metrics(sub['label'], sub['student_pred'], ds))
print_metrics_table(student_metrics, 'STUDENT vs TRUE label on test split')

# (3) Defense C (teacher) vs TRUE label — the upper bound the student could in principle match
teacher_metrics = [per_class_metrics(test_df['label'], test_df['defense_c_label'], 'overall')]
for ds in ['deepset', 'neuralchemy', 'spml']:
    sub = test_df[test_df['dataset']==ds]
    if len(sub) > 0:
        teacher_metrics.append(per_class_metrics(sub['label'], sub['defense_c_label'], ds))
print_metrics_table(teacher_metrics, 'DEFENSE C (teacher) vs TRUE label on test split')

In [ ]:
# Side-by-side comparison
rows = []
for tm, sm in zip(teacher_metrics, student_metrics):
    assert tm['label'] == sm['label']
    rows.append({
        'slice': tm['label'], 'n': tm['n'],
        'teacher_f1': tm['f1'], 'student_f1': sm['f1'], 'delta_f1': sm['f1'] - tm['f1'],
        'teacher_recall': tm['recall'], 'student_recall': sm['recall'],
        'teacher_precision': tm['precision'], 'student_precision': sm['precision'],
    })
compare_df = pd.DataFrame(rows)
print('=== TEACHER (Defense C) vs STUDENT (distilled DistilBERT) on TRUE labels ===')
print(compare_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

## 5. Latency benchmark

Student inference latency, batched and unbatched, vs the Defense C end-to-end timing observed in production. The student's latency is measurable; Defense C's is the sum of (DeBERTa local inference + Sonnet API call), the latter dominated by network round-trip (~2-3 sec per call per §7.4).

In [ ]:
# Unbatched (single-prompt) latency, simulating one-prompt-at-a-time deployment
model.eval()
sample_prompts = test_df['prompt'].sample(20, random_state=SEED).tolist()

# Warm-up
with torch.no_grad():
    for p in sample_prompts[:3]:
        inp = tokenizer(p, return_tensors='pt', truncation=True, max_length=512).to(device)
        _ = model(**inp).logits

import time
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.time()
with torch.no_grad():
    for p in sample_prompts:
        inp = tokenizer(p, return_tensors='pt', truncation=True, max_length=512).to(device)
        _ = model(**inp).logits
torch.cuda.synchronize() if torch.cuda.is_available() else None
single_elapsed = time.time() - t0
single_latency_ms = single_elapsed / len(sample_prompts) * 1000
print(f'Student single-prompt latency: {single_latency_ms:.1f} ms/prompt (n=20)')
print(f'\nDefense C reference (from §7.4): ~6-8 sec per prompt (DeBERTa ~50ms + Sonnet API ~3sec + agent ~5sec for the Defense C decision chain)')
print(f'Student speedup: ~{6000/single_latency_ms:.0f}x to ~{8000/single_latency_ms:.0f}x faster than Defense C')

## 6. Save artifacts

In [ ]:
# Save the student model + tokenizer
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Student saved to {ADAPTER_DIR}')

results = {
    'experiment': 'defense_c_distillation_v1',
    'student_model': STUDENT_MODEL,
    'teacher': 'Defense C = Defense A (ProtectAI DeBERTa) OR Defense B (Sonnet 4.6 v1.21 judge)',
    'training': {
        'epochs': 3, 'lr': 5e-5, 'batch_size': 32, 'seed': SEED,
        'elapsed_min': elapsed / 60,
        'precision': 'bf16' if USE_BF16 else ('fp16' if USE_FP16 else 'fp32'),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    },
    'splits': {s: int((df['split']==s).sum()) for s in ['train', 'val', 'test']},
    'student_vs_teacher': {
        'agreement': float(agree_student_teacher),
        'cohen_kappa': float(kappa_student_teacher),
    },
    'student_metrics': [{**m, 'precision_ci': list(m['precision_ci']), 'recall_ci': list(m['recall_ci'])} for m in student_metrics],
    'teacher_metrics': [{**m, 'precision_ci': list(m['precision_ci']), 'recall_ci': list(m['recall_ci'])} for m in teacher_metrics],
    'comparison_vs_true_labels': compare_df.to_dict('records'),
    'latency_ms_per_prompt': single_latency_ms,
}
with open(RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {RESULTS_PATH}')

## 7. Interpretation guide for §7 deployment writeup

### Student vs teacher (agreement)

| Cohen kappa | Interpretation |
|---|---|
| > 0.80 | Distillation succeeded — student faithfully mimics Defense C. |
| 0.60 - 0.80 | Partial distillation. Student captures most of Defense C's signal but not all. Acceptable for deployment with a small accuracy delta. |
| < 0.60 | Distillation failed. Student is approximating a different function than Defense C. Probably need more training data or a larger student. |

### Student vs true labels (does it actually work?)

| Pattern | Interpretation |
|---|---|
| Student F1 ≈ Teacher F1 (within ±0.02) | Successful distillation: student delivers Defense C's catch rate at DistilBERT's cost and latency. Direct deployment win. |
| Student F1 slightly lower (-0.02 to -0.05) | Modest accuracy cost; acceptable for latency-critical deployments. |
| Student F1 substantially lower (> -0.05) | Distillation didn't preserve enough signal; deploy original Defense C for accuracy-sensitive scenarios. |
| Student F1 HIGHER than teacher (rare) | Student learned to ignore some of Defense C's false positives. Worth investigating. |

### Deployment cost picture

Per 1,000 prompts (from §7.4 numbers):

| Configuration | Latency / prompt | $ / 1,000 prompts |
|---|---|---|
| Defense C (DeBERTa + Sonnet judge) | ~6-8 sec | $2.44 |
| Distilled student (DistilBERT alone) | ~10-50 ms | ~$0 (local GPU inference) |

If the distillation succeeds, the student is **150-800x faster** at **~zero marginal cost** vs Defense C, while preserving the catch rate. That's the deployment artifact for the cost-sensitive scenarios in §7.

## Files produced (on Google Drive)

- `MyDrive/capstone_lora/adapters/distilbert_defense_c_student_v1/` — distilled DistilBERT (~250MB full FT, not LoRA)
- `MyDrive/capstone_lora/results/distillation_metrics.json` — student vs teacher agreement + student/teacher vs true labels

## 8. Robustness sanity checks

Verify the distillation finding before declaring success. Run after section 4 (student_pred and test_df in scope).

### 8.1 Duplicate-prompt check (same as NB08 §10.1)

Memorization risk if train and test share prompts. Reuses the same SPLITS_PATH used by NB08, so this should match.

In [ ]:
import pandas as pd
splits_check = pd.read_parquet(SPLITS_PATH)
train_set = set(splits_check[splits_check['split']=='train']['prompt'])
test_set = set(splits_check[splits_check['split']=='test']['prompt'])
overlap = train_set & test_set
pct = 100 * len(overlap) / len(test_set)
print(f'Exact duplicates train/test: {len(overlap)} of {len(test_set)} ({pct:.2f}%)')

### 8.2 Student vs teacher confusion matrix

Where does the student disagree with Defense C? Are disagreements symmetric or biased?

In [ ]:
from sklearn.metrics import confusion_matrix
cm_st = confusion_matrix(test_df['defense_c_label'], test_df['student_pred'])
print('STUDENT vs TEACHER (Defense C) on test split:')
print(f'                   student=0  student=1')
print(f'  teacher=0       {cm_st[0,0]:>6}    {cm_st[0,1]:>6}')
print(f'  teacher=1       {cm_st[1,0]:>6}    {cm_st[1,1]:>6}')
fpr = cm_st[0,1] / max(cm_st[0,0]+cm_st[0,1], 1)
fnr = cm_st[1,0] / max(cm_st[1,0]+cm_st[1,1], 1)
print(f'\
  Student over-flags teacher CLEAN as FLAGGED: {fpr:.3f}')
print(f'  Student under-flags teacher FLAGGED as CLEAN: {fnr:.3f}')
print(f'\
Interpretation: balanced disagreement (FPR ~ FNR) = symmetric distillation noise.')
print(f'Asymmetric (one direction dominates) = student has a directional bias vs teacher.')

### 8.3 Per-dataset student-teacher agreement

Does distillation fidelity vary by dataset? If kappa is high overall but low on one dataset, the student doesn't generalize the teacher uniformly.

In [ ]:
from sklearn.metrics import cohen_kappa_score
print('Student-teacher kappa per dataset:')
print(f'  {"dataset":<14} {"n":>5} {"agreement":>10} {"kappa":>7}')
for ds in ['deepset', 'neuralchemy', 'spml']:
    sub = test_df[test_df['dataset']==ds]
    if len(sub) > 0:
        agree = (sub['defense_c_label'] == sub['student_pred']).mean()
        k = cohen_kappa_score(sub['defense_c_label'], sub['student_pred'])
        print(f'  {ds:<14} {len(sub):>5} {agree:>10.3f} {k:>7.3f}')

### 8.4 Prompt-length shortcut check

Does the student learn 'long prompt → flagged'?

In [ ]:
test_df_lc = test_df.copy()
test_df_lc['prompt_len'] = test_df_lc['prompt'].str.len()
print('Prompt length by student prediction:')
print(test_df_lc.groupby('student_pred')['prompt_len'].agg(['count', 'mean', 'median']).round(0))
print()
ratio = (test_df_lc[test_df_lc['student_pred']==1]['prompt_len'].mean() /
         max(test_df_lc[test_df_lc['student_pred']==0]['prompt_len'].mean(), 1))
status = 'SHORTCUT RISK' if ratio > 2.0 or ratio < 0.5 else 'OK'
print(f'Ratio student_pred=1 / student_pred=0 length: {ratio:.2f}x  [{status}]')